In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Run Bronze

- Bronze ingestion (simple batch approach)
- Discovers new Excel files in the landing zone, 
- Parse sheets via ExcelSheetParser
- Write to Bronze table
- Moves processed files to archive



In [0]:
import os
import uuid
import shutil
from pyspark.sql import functions as F

from utils.excel_parser import ExcelSheetParser
from utils.logger import get_logger
from utils.summary_report import create_summary_report
from etl_config.bronze_config import (
    CATALOG,
    BRONZE,
    BRONZE_CONFIG,
    BRONZE_SCHEMA,
    LANDING_DIR,
    PROCESSED_DIR,
    FAILED_DIR,
)

logger = get_logger("bronze_ingestion")

# Create processed and failed directories if they dont exists
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FAILED_DIR, exist_ok=True)

### Step 1: Get all unprocessed .xlsx files

In [0]:
# List of file paths
landing_files = [
    os.path.join(LANDING_DIR, f)
    for f in os.listdir(LANDING_DIR)
    if f.endswith(".xlsx") and os.path.isfile(os.path.join(LANDING_DIR, f))
]

if not landing_files:
    logger.info("No new Excel files found in landing zone. Nothing to do.")
    dbutils.jobs.taskValues.set(key="exit_value", value="NO_NEW_FILES")
    dbutils.notebook.exit("NO_NEW_FILES")

logger.info(f"Found {len(landing_files)} file(s) to process: {landing_files}")

### Step 2: Process each file sequntially

In [0]:
failed_files = []
batch_ids = []
for file_path in landing_files:
    file_name = os.path.basename(file_path)
    batch_id = str(uuid.uuid4())
    logger.info(f"Processing file: {file_name} (batch_id={batch_id})")

    try:
        parser = ExcelSheetParser(file_path)
        tables = parser.parse_all_sheets()
        logger.info(f"Discovered tables: {list(tables.keys())}")

        for table_name, pdf in tables.items():
            
            sdf = spark.createDataFrame(pdf)
            sdf = (
                sdf.withColumn("_source_file", F.lit(parser.source_file_name))
                   .withColumn("_ingested_at", F.current_timestamp())
                   .withColumn("_batch_id", F.lit(batch_id))
            )

            full_table_name = f"{BRONZE}.{table_name}"

            (
                sdf.write
                   .format("delta")
                   .mode("append")
                   .option("mergeSchema", "true")
                   .saveAsTable(full_table_name)
            )

            logger.info(f"OK: {full_table_name} -> appended {sdf.count()} rows")

        # move to processed only after all sheets succeed
        dest = os.path.join(PROCESSED_DIR, file_name)
        shutil.move(file_path, dest)
        logger.info(f"Moved processed file to: {dest}")
        
        batch_ids.append(batch_id)
    except Exception as e:
        logger.error(f"Failed to process {file_name}: {e}")
        dest = os.path.join(FAILED_DIR, file_name)
        shutil.move(file_path, dest)
        logger.error(f"Moved failed file to: {dest}")
        failed_files.append(file_name)

if failed_files:
    logger.error(f"Batch completed with failures: {failed_files}")
    dbutils.jobs.taskValues.set(key="exit_value", value="PARTIAL_FAILURE")
else:
    dbutils.jobs.taskValues.set(key="exit_value", value="SUCCESS")
    logger.info("Bronze ingestion complete for all discovered files")

if failed_files and not batch_ids:
    raise RuntimeError(f"Bronze ingestion failed for all {len(failed_files)} file(s): {failed_files}")


In [0]:
if batch_ids:
    summary = create_summary_report(
        spark,
        layer_name=BRONZE_SCHEMA,
        tables={name: {} for name in BRONZE_CONFIG.keys()},
        catalog=CATALOG,
        schema=BRONZE_SCHEMA,
        batch_ids=batch_ids,
    )